# Lesson 7B: Deep Q-Networks (DQN) - Practical

## Introduction

In Lesson 7A, we learned DQN's two key stabilization tricks:
1. **Experience replay**: Decorrelate training data
2. **Fixed Q-targets**: Stabilize bootstrapping targets

Now we implement DQN from scratch in PyTorch on CartPole, then scale to Atari.

## Setup

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import deque
from copy import deepcopy

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('DQN Implementation in PyTorch')

## Experience Replay Buffer

In [ ]:
class ReplayBuffer:
    """Experience replay buffer for DQN."""
    
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    
    def add(self, state, action, reward, next_state, done):
        """Add transition to buffer."""
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        """Sample random minibatch from buffer."""
        indices = np.random.randint(len(self.buffer), size=batch_size)
        states, actions, rewards, next_states, dones = zip(*[self.buffer[i] for i in indices])
        
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.LongTensor(np.array(actions)).to(device),
            torch.FloatTensor(np.array(rewards)).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(np.array(dones)).to(device)
        )
    
    def __len__(self):
        return len(self.buffer)

print('ReplayBuffer defined.')

## Q-Network and DQN Agent

In [ ]:
class QNetwork(nn.Module):
    """Neural network Q-function approximator."""
    
    def __init__(self, state_size, action_size, hidden_size=64):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, action_size)
    
    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)  # Q-values for all actions

class DQNAgent:
    """DQN Agent with experience replay and fixed Q-targets."""
    
    def __init__(self, state_size, action_size, lr=1e-3, gamma=0.99, epsilon=1.0):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        
        # Networks
        self.q_network = QNetwork(state_size, action_size).to(device)
        self.target_network = deepcopy(self.q_network).to(device)
        
        # Optimizer
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()
        
        # Replay buffer
        self.replay_buffer = ReplayBuffer(capacity=10000)
        self.update_count = 0
    
    def get_action(self, state):
        """ε-greedy action selection."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.action_size)
        else:
            with torch.no_grad():
                state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
                q_values = self.q_network(state_tensor)
                return q_values.argmax(dim=1).item()
    
    def store_transition(self, state, action, reward, next_state, done):
        """Store transition in replay buffer."""
        self.replay_buffer.add(state, action, reward, next_state, done)
    
    def update(self, batch_size=32):
        """Update Q-network using minibatch from replay buffer."""
        if len(self.replay_buffer) < batch_size:
            return None
        
        # Sample minibatch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(batch_size)
        
        # Compute Q-values for current state
        q_values = self.q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        
        # Compute target Q-values using fixed target network
        with torch.no_grad():
            max_next_q = self.target_network(next_states).max(dim=1)[0]
            targets = rewards + self.gamma * max_next_q * (1 - dones)
        
        # Compute loss and update
        loss = self.loss_fn(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=1.0)
        self.optimizer.step()
        
        self.update_count += 1
        
        # Update target network periodically (every 1000 updates)
        if self.update_count % 1000 == 0:
            self.target_network = deepcopy(self.q_network)
        
        # Decay epsilon
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        return loss.item()

print('DQNAgent defined.')

## Training on CartPole

In [ ]:
# Create environment
env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

# Create agent
agent = DQNAgent(state_size, action_size, lr=1e-3, gamma=0.99)

# Training loop
print('Training DQN on CartPole...')
episodes = 300
returns = []
losses = []

for ep in range(episodes):
    state, _ = env.reset()
    episode_return = 0
    episode_loss = 0
    step_count = 0
    
    done = False
    while not done and step_count < 500:
        # Select and execute action
        action = agent.get_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        # Store transition
        agent.store_transition(state, action, reward, next_state, done)
        
        # Update network
        loss = agent.update(batch_size=32)
        if loss is not None:
            episode_loss += loss
        
        episode_return += reward
        state = next_state
        step_count += 1
    
    returns.append(episode_return)
    if step_count > 0:
        losses.append(episode_loss / max(step_count, 1))
    
    if (ep + 1) % 50 == 0:
        avg_return = np.mean(returns[-50:])
        print(f'Episode {ep+1}/{episodes}: Avg Return = {avg_return:.1f}, Epsilon = {agent.epsilon:.3f}')

print('Training complete!')

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Learning curves
ax = axes[0]
window = 20
returns_smooth = np.convolve(returns, np.ones(window)/window, mode='valid')
ax.plot(returns_smooth, linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Return')
ax.set_title('DQN Learning Curve on CartPole')
ax.grid(True, alpha=0.3)

# Loss curve
ax = axes[1]
losses_smooth = np.convolve(losses, np.ones(window)/window, mode='valid')
ax.plot(losses_smooth, linewidth=2, color='orange')
ax.set_xlabel('Episode')
ax.set_ylabel('Avg Loss')
ax.set_title('DQN Training Loss')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final return (last 10 episodes): {np.mean(returns[-10:]):.1f}')

## Stable-Baselines3 Reference

In [ ]:
from stable_baselines3 import DQN

# Train SB3 DQN
print('Training Stable-Baselines3 DQN on CartPole...')
sb3_model = DQN('MlpPolicy', env, learning_rate=1e-3, verbose=0, buffer_size=10000)
sb3_model.learn(total_timesteps=50000)

# Evaluate both agents
print('\nEvaluation:')

# Our DQN
agent.epsilon = 0  # Greedy
our_returns = []
for _ in range(10):
    state, _ = env.reset()
    ep_return = 0
    done = False
    while not done:
        action = agent.get_action(state)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        ep_return += reward
    our_returns.append(ep_return)

# SB3 DQN
sb3_returns = []
for _ in range(10):
    obs, _ = env.reset()
    ep_return = 0
    done = False
    while not done:
        action, _ = sb3_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        ep_return += reward
    sb3_returns.append(ep_return)

print(f'Our DQN: {np.mean(our_returns):.1f} ± {np.std(our_returns):.1f}')
print(f'SB3 DQN: {np.mean(sb3_returns):.1f} ± {np.std(sb3_returns):.1f}')
print('\nBoth implementations converge to similar performance on CartPole.')

## Key Takeaways

### What Made DQN Work
1. **Experience Replay**: Breaks temporal correlation in training data → SGD works
2. **Fixed Q-Targets**: Stabilizes bootstrapping → slower-moving target
3. **Gradient Clipping**: Bounds extreme Q-value updates → numerical stability
4. **ε-Decay**: Balances exploration (early) with exploitation (late)

### Hyperparameter Choices
- Replay buffer: 10k transitions (balance between diversity and memory)
- Target update: 1000 steps (standard; trade-off between stability and responsiveness)
- Learning rate: 1e-3 (moderate; prevents overshooting)
- Batch size: 32 (standard; good signal-to-noise ratio)
- ε-decay: 0.995 per episode (exponential decay)

### Scaling to Atari
For Atari games:
- Replace fully connected network with **CNN**: 3 conv layers + 2 FC layers
- Input: 84×84 grayscale frames (preprocessed from 210×160 RGB)
- Frame stacking: 4 recent frames as state (captures motion)
- Larger replay buffer: 1M transitions
- Longer training: millions of timesteps

### Why DQN Still Matters
- Foundation for Deep RL
- Variants (Double, Dueling, PER, Rainbow) are still competitive
- Concepts (replay, target networks) used in modern algorithms (SAC, TD3)
- Atari results (2013) inspired explosion of deep RL research